# 03 — Build customer measures

**Purpose.** Turn each customer's purchase history into measures a model can learn from, one row per
customer per cutoff date, using only purchases before that date. Logic lives in `src/features.py`,
with tests in `tests/test_features.py` (including a check that future purchases cannot change any
measure); this notebook runs it and looks at how each measure relates to churn.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data import PROCESSED_DIR, load_clean
from src.features import build_features
from src.labels import build_snapshots, split_cutoffs, valid_cutoffs

pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Build the measures

Same churn labels as step 2 (90-day window, 10 cutoffs), with the measures added. Each group of
measures, as defined in `src/features.py`:

* **Recency, frequency, spend:** days since last order; orders and spend in the last 30, 90, 180 and 365 days.
* **Trend:** spend in the last 90 days against the 90 before, from −1 (stopped) to +1 (new or growing).
* **Buying rhythm:** usual gap between order dates, how much it varies, and how overdue the customer is
  (days since last order ÷ usual gap). One-time buyers have no gap, so these are left empty and flagged.
* **Tenure, basket, returns:** days since first order; average order value and products per order;
  cancelled value as a share of spend.
* **Season and location:** ordered in September–November in the past year; mostly buys from the UK.

Product-mix measures wait until product categories are decided. The result is saved to
`data/processed/features.parquet` for steps 4 and 5.

In [2]:
data = load_clean()
cutoffs = valid_cutoffs(90)
train_cuts, test_cuts = split_cutoffs(cutoffs, horizon_days=90, n_test=3)

snapshots = build_snapshots(data.purchases, cutoffs, horizon_days=90)
features = build_features(data.purchases, data.cancellations, snapshots)
features.to_parquet(PROCESSED_DIR / "features.parquet", index=False)

MEASURES = [c for c in features.columns if c not in ("customer_id", "cutoff", "horizon_days", "churn")]
print(f"{len(features):,} rows x {len(MEASURES)} measures")
features[MEASURES].isna().sum().loc[lambda s: s > 0].rename("missing (one-time buyers etc.)")

42,850 rows x 21 measures


median_gap_days    12385
gap_std_days       20321
overdue_ratio      12385
Name: missing (one-time buyers etc.), dtype: int64

## 2. Churn rate by measure (training cutoffs only)

For each measure, customers are split into five equal groups from lowest (Q1) to highest (Q5) value,
and the table shows the churn rate in each group; yes/no measures show the rate for 0 and 1. A useful
measure shows a clear rise or fall across the groups. Only the training cutoffs are used: choosing
measures by how they look on the test months would quietly leak test information into the model.

In [3]:
train = features[features["cutoff"].isin(train_cuts)]

def churn_by_group(col):
    x = train[col].dropna()
    if x.nunique() <= 2:
        return train.groupby(col)["churn"].mean().rename(lambda v: f"={v:g}")
    groups = pd.qcut(x.rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
    return train.loc[x.index].groupby(groups, observed=True)["churn"].mean()

pd.DataFrame({c: churn_by_group(c) for c in MEASURES}).T

,=0,=1,Q1,Q2,Q3,Q4,Q5
recency_days,NaN,NaN,0.40,0.56,0.67,0.76,0.85
tenure_days,NaN,NaN,0.75,0.76,0.70,0.57,0.47
orders_30d,NaN,NaN,0.75,0.73,0.70,0.68,0.39
spend_30d,NaN,NaN,0.75,0.73,0.70,0.68,0.38
orders_90d,NaN,NaN,0.82,0.76,0.74,0.60,0.31
spend_90d,NaN,NaN,0.82,0.76,0.73,0.60,0.33
orders_180d,NaN,NaN,0.85,0.80,0.73,0.60,0.27
spend_180d,NaN,NaN,0.85,0.80,0.71,0.58,0.30
orders_365d,NaN,NaN,0.87,0.81,0.73,0.57,0.26
spend_365d,NaN,NaN,0.86,0.81,0.72,0.56,0.30


## 3. Does "overdue" add anything beyond order count?

Order count alone already separates churners well (step 2). Holding it roughly fixed, at 3–5 orders in
the past year, churn should still rise for customers who are far past their usual gap if the rhythm
measures carry extra information.

In [4]:
mid = train[train["orders_365d"].between(3, 5)].dropna(subset=["overdue_ratio"])
bands = pd.cut(mid["overdue_ratio"], [0, 0.5, 1, 2, 4, float("inf")],
               labels=["< 0.5x", "0.5-1x", "1-2x", "2-4x", "> 4x"])
mid.groupby(bands, observed=True)["churn"].agg(snapshots="size", churn_rate="mean")

,snapshots,churn_rate
overdue_ratio,,
< 0.5x,1564,0.58
0.5-1x,1145,0.55
1-2x,1313,0.58
2-4x,828,0.63
> 4x,605,0.75


## 4. Findings

1. **Recency and frequency are the strongest single measures.** Churn rises from 40% for the most
   recent buyers to 85% for the least recent, and falls from about 87% to 26% across order-count groups.
   Spend in each window behaves almost exactly like the order count, so the two say much the same thing.
2. **Buying rhythm adds information.** Among customers with 3–5 orders in the past year, churn rises
   from about 55% for those on schedule to 75% for those more than four times past their usual gap.
3. **Trend is not a straight line.** Both ends (−1 and +1) and the middle (0, no spend in either
   period) churn more than the groups in between, because the extremes are mostly infrequent buyers.
   Models that handle non-linear patterns (gradient boosting) should use it better than a straight-line model.
4. **Season matters.** Customers who bought in the past autumn churn 58% of the time against 80%
   for those who did not.
5. **Weak measures:** products per order and UK vs. other show little difference; keep them for now
   and let step 6 show whether they add anything.